In [ ]:
from ultralytics import YOLO
from ultralytics import settings
from ultralytics.data.split import autosplit
from ultralytics.utils.plotting import Annotator
import json
import os
import sys
import cv2
import numpy as np
from datetime import datetime
from pathlib import Path
import matplotlib.pyplot as plt
import supervision as sv

from utilities.annotation_converter import AnnotationConverter
from utilities.visualization import VideoAnnotator

In [ ]:
class YOLODetector:
    def __init__(self,
                 model_name="yolo11n.pt",
                 experiment_name="yolo11_h23",
                 data_config="../../data/h23/h23.yaml",
                 data_dir="../../data/",
                 runs_dir="../../outputs/runs",
                 outputs_dir="../../outputs",
                 _experiment_dir=None):
        self.model_name = model_name
        self.model = YOLO(model_name)
        self.experiment_name = experiment_name
        self.data_config = Path(data_config).resolve()
        self.data_dir = Path(data_dir).resolve()
        self.runs_dir = Path(runs_dir).resolve()
        self.runs_dir.mkdir(parents=True, exist_ok=True)

        # Experiment directory: reuse existing if restoring, else create timestamped one
        if _experiment_dir is not None:
            self.experiment_dir = Path(_experiment_dir).resolve()
        else:
            timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
            self.experiment_dir = (
                Path(outputs_dir).resolve() / "experiments" / f"{experiment_name}_{timestamp}"
            )
            self.experiment_dir.mkdir(parents=True, exist_ok=True)
            config = {
                "model_name": model_name,
                "experiment_name": experiment_name,
                "data_config": str(self.data_config),
                "created_at": datetime.now().isoformat(),
            }
            (self.experiment_dir / "config.json").write_text(json.dumps(config, indent=2))

        settings.update({"runs_dir": str(self.runs_dir),
                         "datasets_dir": str(self.data_dir),
                         "tensorboard": False,
                         "wandb": True})
        print(f"Model type: {self.model.model_name}")
        print(f"Using data config: {self.data_config}")
        print(f"Runs dir:          {self.runs_dir}")
        print(f"Experiment dir:    {self.experiment_dir}")

    # ------------------------------------------------------------------
    # Training
    # ------------------------------------------------------------------

    def train(self, epochs=1, imgsz=640, name=None):
        if name is None:
            name = f"{self.experiment_name}_train"
        self.model.train(
            data=self.data_config,
            epochs=epochs,
            imgsz=imgsz,
            project=str(self.runs_dir),
            name=name,
        )
        self.save_model()

    # ------------------------------------------------------------------
    # Model saving / loading
    # ------------------------------------------------------------------

    def save_model(self):
        """Write a model.json into the experiment dir pointing at the trained weights."""
        if self.model.trainer is None:
            raise RuntimeError("No training run found. Call train() first.")
        run_dir = Path(self.model.trainer.save_dir)
        best_pt = run_dir / "weights" / "best.pt"
        last_pt = run_dir / "weights" / "last.pt"
        model_info = {
            "run_dir": str(run_dir),
            "best_weights": str(best_pt) if best_pt.exists() else None,
            "last_weights": str(last_pt) if last_pt.exists() else None,
        }
        out = self.experiment_dir / "model.json"
        out.write_text(json.dumps(model_info, indent=2))
        print(f"Model info saved to: {out}")
        print(f"  best.pt  → {model_info['best_weights']}")
        print(f"  last.pt  → {model_info['last_weights']}")
        return best_pt

    def load_best_weights(self):
        """Replace self.model with the best trained weights for this experiment."""
        model_json = self.experiment_dir / "model.json"
        if not model_json.exists():
            raise FileNotFoundError(f"No model.json in {self.experiment_dir}")
        info = json.loads(model_json.read_text())
        best_pt = info.get("best_weights")
        if not best_pt or not Path(best_pt).exists():
            raise FileNotFoundError(f"best.pt not found: {best_pt}")
        self.model = YOLO(best_pt)
        print(f"Loaded best weights: {best_pt}")

    # ------------------------------------------------------------------
    # Inference
    # ------------------------------------------------------------------

    def detect(self, test_data_path):
        results = self.model.predict(test_data_path)
        return results

    # ------------------------------------------------------------------
    # Detection results saving / loading
    # ------------------------------------------------------------------

    def save_detections(self, results) -> dict[str, sv.Detections]:
        """Serialise YOLO results to <experiment_dir>/detections.json.

        Returns the predictions dict (frame stem → sv.Detections) so it can be
        used directly in the visualisation step.
        """
        predictions: dict[str, sv.Detections] = {}
        serialisable: dict[str, dict] = {}
        for result in results:
            stem = Path(result.path).stem
            xyxy = result.boxes.xyxy.cpu().numpy()
            conf = result.boxes.conf.cpu().numpy()
            cls  = result.boxes.cls.cpu().numpy().astype(int)
            predictions[stem] = sv.Detections(xyxy=xyxy, confidence=conf, class_id=cls)
            serialisable[stem] = {
                "xyxy":       xyxy.tolist(),
                "confidence": conf.tolist(),
                "class_id":   cls.tolist(),
            }
        out = self.experiment_dir / "detections.json"
        out.write_text(json.dumps(serialisable, indent=2))
        print(f"Detections saved to: {out}  ({len(predictions)} frames)")
        return predictions

    def load_detections(self) -> dict[str, sv.Detections]:
        """Load detections from <experiment_dir>/detections.json."""
        det_path = self.experiment_dir / "detections.json"
        if not det_path.exists():
            raise FileNotFoundError(f"No detections.json in {self.experiment_dir}")
        raw = json.loads(det_path.read_text())
        predictions = {
            stem: sv.Detections(
                xyxy=np.array(d["xyxy"],       dtype=np.float32),
                confidence=np.array(d["confidence"], dtype=np.float32),
                class_id=np.array(d["class_id"],  dtype=int),
            )
            for stem, d in raw.items()
        }
        print(f"Loaded detections for {len(predictions)} frames from: {det_path}")
        return predictions

    # ------------------------------------------------------------------
    # Restore from saved experiment
    # ------------------------------------------------------------------

    @classmethod
    def from_experiment(cls, experiment_dir: str) -> "YOLODetector":
        """Restore a YOLODetector from a previously saved experiment directory.

        Loads config.json for hyperparameters and model.json for weights.
        The restored detector points at the same experiment_dir so
        load_detections() works immediately.
        """
        exp_dir = Path(experiment_dir).resolve()
        config_path = exp_dir / "config.json"
        if not config_path.exists():
            raise FileNotFoundError(f"No config.json in {experiment_dir}")
        config = json.loads(config_path.read_text())

        # Prefer trained best.pt; fall back to original pretrained weights
        model_json = exp_dir / "model.json"
        if model_json.exists():
            info = json.loads(model_json.read_text())
            best_pt = info.get("best_weights")
            model_name = best_pt if best_pt and Path(best_pt).exists() else config["model_name"]
        else:
            model_name = config["model_name"]

        detector = cls(
            model_name=model_name,
            experiment_name=config["experiment_name"],
            data_config=config["data_config"],
            _experiment_dir=str(exp_dir),
        )
        print(f"Restored from experiment: {exp_dir}")
        return detector

In [3]:
detector = YOLODetector()

Model type: yolo11n.pt
Using data config /storage/ice1/3/2/rrivera73/freeman-bird-detection/data/h23/h23.yaml
Runs will be saved to: /storage/ice1/3/2/rrivera73/freeman-bird-detection/outputs/runs/yolo11_h23


In [4]:
detector.train()

New https://pypi.org/project/ultralytics/8.4.21 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.14 🚀 Python-3.11.14 torch-2.10.0+cu128 CUDA:0 (NVIDIA A40, 45488MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/storage/ice1/3/2/rrivera73/freeman-bird-detection/data/h23/h23.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=1, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /home/hice1/rrivera73/.netrc.
wandb: Currently logged in as: beccaarivera (tigermalk) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Overriding model.yaml nc=80 with nc=1

                   from  n    params  module                                       arguments                     
  0                  -1  1       464  ultralytics.nn.modules.conv.Conv             [3, 16, 3, 2]                 
  1                  -1  1      4672  ultralytics.nn.modules.conv.Conv             [16, 32, 3, 2]                
  2                  -1  1      6640  ultralytics.nn.modules.block.C3k2            [32, 64, 1, False, 0.25]      
  3                  -1  1     36992  ultralytics.nn.modules.conv.Conv             [64, 64, 3, 2]                
  4                  -1  1     26080  ultralytics.nn.modules.block.C3k2            [64, 128, 1, False, 0.25]     
  5                  -1  1    147712  ultralytics.nn.modules.conv.Conv             [128, 128, 3, 2]              
  6                  -1  1     87040  ultralytics.nn.modules.block.C3k2            [128, 128, 1, True]           
  7                  -1  1    295424  ultralytics

/home/hice1/rrivera73/.conda/envs/birdenv/lib/python3.11/site-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 16 worker processes in total. Our suggested max number of worker in current system is 12, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/home/hice1/rrivera73/.conda/envs/birdenv/lib/python3.11/site-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 16 worker processes in total. Our suggested max number of worker in current system is 12, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_work

optimizer: 'optimizer=auto' found, ignoring 'lr0=0.01' and 'momentum=0.937' and determining best 'optimizer', 'lr0' and 'momentum' automatically... 
optimizer: AdamW(lr=0.002, momentum=0.9) with parameter groups 81 weight(decay=0.0), 88 weight(decay=0.0005), 87 bias(decay=0.0)
Plotting labels to /storage/ice1/3/2/rrivera73/freeman-bird-detection/outputs/runs/yolo11_h23_train3/labels.jpg... 
Image sizes 640 train, 640 val
Using 8 dataloader workers
Logging results to /storage/ice1/3/2/rrivera73/freeman-bird-detection/outputs/runs/yolo11_h23_train3
Starting training for 1 epochs...

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
        1/1      2.32G      2.284      4.173      1.373          6        640: 100% ━━━━━━━━━━━━ 1212/1212 3.2it/s 6:16<0.2s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 152/152 5.0it/s 30.2s0.1ss
                   all       4843       1904      0.546      0.363      0.

wandb: WARNING Tried to log to step 1 that is less than the current step 2. Steps must be monotonically increasing, so this data will be ignored. See https://wandb.me/define-metric to log data out of order.


lr/pg0,▁
lr/pg1,▁
lr/pg2,▁
metrics/mAP50(B),▁
metrics/mAP50-95(B),▁
metrics/precision(B),▁
metrics/recall(B),▁
model/GFLOPs,▁
model/parameters,▁
model/speed_PyTorch(ms),▁
+6,...


In [6]:
results = detector.detect("../../data/h23/images/test")


WARNING ⚠️ 
Inference results will accumulate in RAM unless `stream=True` is passed, which can cause out-of-memory errors for large
sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict/ for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs

image 1/3636 /storage/ice1/3/2/rrivera73/freeman-bird-detection/models/yolo/../../data/h23/images/test/IMG_0074_frame_000000.png: 384x640 (no detections), 120.4ms
image 2/3636 /storage/ice1/3/2/rrivera73/freeman-bird-detection/models/yolo/../../data/h23/images/test/IMG_0074_frame_000001.png: 384x640 (no detections), 9.9ms
image 3/3636 /storage/ice1/3/2/rrivera73/freeman-bird-detection/models/yolo/../../data/h23/images/test/IMG_0074_frame_000002.png: 384x

In [7]:
boxes = results[0].boxes
print(boxes.xyxy[0]) 

IndexError: index 0 is out of bounds for dimension 0 with size 0

## Visualization: GT vs. Predicted Bounding Boxes

In [8]:
sys.path.append("../../")

CONFIG = {
    "data_yaml":    "../../data/h23/h23.yaml",
    "images_dir":   "../../data/h23/images/test",
    "labels_dir":   "../../data/h23/labels/test",   # set to None for pred-only
    "output_path":  "../../outputs/annotated_video.mp4",
    "fps":          29.0,
}


In [ ]:
# Save detections to experiment_dir/detections.json and return the predictions dict
predictions_dict = detector.save_detections(results)

In [ ]:
annotator = VideoAnnotator(
    data_yaml_path=CONFIG["data_yaml"],
    images_dir=CONFIG["images_dir"],
    labels_dir=CONFIG["labels_dir"],   # set to None for pred-only mode
    predictions=predictions_dict,       # set to dict for pred or both mode
)

annotator.annotate_video(output_path=CONFIG["output_path"], fps=CONFIG["fps"])
print(f"Annotated video written to: {CONFIG['output_path']}")


SupervisionWarnings: BoundingBoxAnnotator is deprecated: `BoundingBoxAnnotator` is deprecated and has been renamed to `BoxAnnotator`. `BoundingBoxAnnotator` will be removed in supervision-0.26.0.
SupervisionWarnings: BoundingBoxAnnotator is deprecated: `BoundingBoxAnnotator` is deprecated and has been renamed to `BoxAnnotator`. `BoundingBoxAnnotator` will be removed in supervision-0.26.0.
GT label file missing for stem 'IMG_0074_frame_000000'
GT label file missing for stem 'IMG_0074_frame_000001'
GT label file missing for stem 'IMG_0074_frame_000002'
GT label file missing for stem 'IMG_0074_frame_000003'
GT label file missing for stem 'IMG_0074_frame_000004'
GT label file missing for stem 'IMG_0074_frame_000005'
GT label file missing for stem 'IMG_0074_frame_000006'
GT label file missing for stem 'IMG_0074_frame_000007'
GT label file missing for stem 'IMG_0074_frame_000008'
GT label file missing for stem 'IMG_0074_frame_000009'
GT label file missing for stem 'IMG_0074_frame_000010'
GT 

In [22]:
import matplotlib.pyplot as plt
import cv2
from pathlib import Path

# Pick any frame from images_dir to spot-check
sample_frames = sorted(Path(CONFIG["images_dir"]).glob("*.jpg"))
if not sample_frames:
    sample_frames = sorted(Path(CONFIG["images_dir"]).glob("*.png"))

if sample_frames:
    frame_path = str(sample_frames[0])
    annotated = annotator.annotate_single_frame(frame_path)
    annotated_rgb = cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB)

    plt.figure(figsize=(14, 9))
    plt.imshow(annotated_rgb)
    plt.title(f"Spot-check: {Path(frame_path).name}")
    plt.axis("off")
    plt.tight_layout()
    plt.show()
else:
    print("No frames found in", CONFIG["images_dir"])


<Figure size 1400x900 with 1 Axes>

## Loading a Saved Experiment

Use `YOLODetector.from_experiment()` to restore a detector (with trained weights) from any
previously saved experiment directory, then call `load_detections()` to recover the cached
inference results — no retraining or re-inference needed.

In [ ]:
# ── Restore from a saved experiment ──────────────────────────────────────────
# Replace the path below with the actual experiment directory printed during init.
# Example: "../../outputs/experiments/yolo11_h23_20260313_120000"

# loaded_detector = YOLODetector.from_experiment(str(detector.experiment_dir))
# loaded_predictions = loaded_detector.load_detections()

# List all saved experiments
experiments_root = Path("../../outputs/experiments")
if experiments_root.exists():
    experiments = sorted(experiments_root.iterdir())
    print(f"Saved experiments ({len(experiments)}):")
    for exp in experiments:
        config_path = exp / "config.json"
        has_model = (exp / "model.json").exists()
        has_dets  = (exp / "detections.json").exists()
        tag = f"[model={'✓' if has_model else '✗'}  dets={'✓' if has_dets else '✗'}]"
        print(f"  {exp.name}  {tag}")
else:
    print("No experiments saved yet.")